In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

/Users/ayhan/Desktop/DS_Bootcamp/Capstone_Martin_Ayhan/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### Wir bauen nun einen zweiten Embedding Schritt. Der erste Embedding Schritt war für die Issue_Detection. Ziel war es dabei Clustering zu machen. Der zweite Schritt des Embeddings ist für Semantic Search, RAG und AI Insights.

In [2]:
#Reviews laden
import pandas as pd

df=pd.read_csv("../data/BMW/clean_reviews.csv")
texts = df["content"].astype(str).tolist()
print("Number of reviews:", len(texts))

Number of reviews: 10509


In [3]:
#Embeddings erstellen
embeddings = model.encode(texts, batch_size=32, show_progress_bar = True)
print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 329/329 [00:33<00:00,  9.85it/s]

Embeddings shape: (10509, 384)


In [4]:
#FAISS Index bauen
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))
print("Total vectors in index:", index.ntotal)

Total vectors in index: 10509


In [5]:
#Semantic Search testen
query = "BMW app crashes when opening vehicle status"
query_embedding = model.encode([query])
distances, indices = index.search(query_embedding, k=5)
df.iloc[indices[0]][["content"]]

,content
329,"since last update, app crashes at login with n..."
9753,app kann nicht mehr auf fahrzeugstatus zugreif...
3595,my car disappeared from the app. tried adding ...
85,at 1st i thought that there is a problem with ...
3330,app is not working properly. vehicle finder an...


In [6]:
#Index speichern
faiss.write_index(index, "../models/review_index.faiss")

In [7]:
#Embeddings speichern
np.save("../models/review_embeddings.npy", embeddings)